# Day 10 · 数据清洗与去重

**配套讲义**: `days/day-10.md` ｜ **本地可跑**
W2 最重要的一天。每一条被删的数据，你都要能说出为什么。

## 1. 五段清洗流水线全量跑

In [ ]:
import sys; sys.path.insert(0, "..")
from src.data.dedup import clean_pipeline

inp = "../data/synthetic/sft_2k.jsonl"     # 没有就先用 pilot.jsonl
out = "../data/clean/"
import os; os.makedirs(out, exist_ok=True)
try:
    report = clean_pipeline(inp, out)
    print(report.to_markdown())            # CleaningReport
except FileNotFoundError:
    print("找不到", inp, "—— 先跑 Day 9 的正式合成，或把 inp 改成 pilot 文件。")

## 2. 检查「同图不同问」被保留

In [ ]:
from src.data.dedup import dedup_simple
# 构造：同一张图、两个不同问题 —— 正当样本，不能被图像去重误杀
same_img_diff_q = [
    {"id": "a1", "image": "imgA", "question": "这件多大码？", "answer": "M/L 有货。"},
    {"id": "a2", "image": "imgA", "question": "什么材质？",   "answer": "95% 棉。"},
    {"id": "a3", "image": "imgA", "question": "这件多大码？", "answer": "M/L 有货。"},  # 真·重复
]
kept, dropped = dedup_simple(same_img_diff_q)
print("保留:", [r["id"] for r in kept])      # 期望 a1, a2
print("淘汰:", [r["id"] for r in dropped])   # 期望 a3

## 3. 隔离区抽检（quarantine）

In [ ]:
import json, random
from pathlib import Path
q = Path("../data/clean/quarantine.jsonl")
if q.exists():
    rows = [json.loads(l) for l in q.read_text().splitlines() if l.strip()]
    for r in random.sample(rows, min(5, len(rows))):
        print("-" * 50)
        print("淘汰原因:", r.get("drop_reason"))
        print("内容摘要:", str(r)[:120])
    print("\n→ 误杀率感受一下。太高（>30%）就回去调阈值。")
else:
    print("流水线跑完才会有 quarantine.jsonl")

## 4. 思考题自答（写在打卡里）
1. 字面去重漏什么？语义去重误杀什么？
2. 为什么按图去重而不是按样本？
3. 空回答样本删还是留？

## 5. 验收
- [ ] 报告每条淘汰有原因字段
- [ ] 语义去重误杀率 < 30%
- [ ] `data/clean/REPORT.md` 已生成